In [1]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.external_data_encyclopedia import ExternalDataEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

ed = ExternalDataEncyclopedia(
    external_data_path=Path(r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data")
)

In [2]:
exports_folder = Path("refine_plus_export_pool")

In [ ]:
from pathlib import Path
from typing import List, Dict

import igl
import numpy as np
import polars as pl
from polars import Series
from polars.dataframe.frame import DataFrame

from boulder_statistics.refinement_plus.facet_parser import FacetParser
from boulder_statistics.refinement_plus.bulk_parse_data_tir_maps import (
    FACET_SHAPE_MODELS,
    TIR_MEASUREMENT_NAMES,
    TIR_SIGMA_MEASUREMENT_NAMES,
    DataTirMaps,
)
from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from boulder_statistics.refinement_plus.refinement_chunking import ChunkingTools


mesh_folder = Path(
    r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data\bennu_models"
)

measurement_types: list[str] = ["tri_num"]

# TIR_MEASUREMENT_NAMES + TIR_SIGMA_MEASUREMENT_NAMES + 

data_tir_maps: DataFrame = DataTirMaps.bulk_parse(ed)
mission_phases: Series = data_tir_maps["mission_phase"].unique()


for mission_phase in mission_phases:
    tir_spectral_export_path: Path = exports_folder / f"01_add_tir_maps_{mission_phase}"
    if tir_spectral_export_path.exists():
        continue

    print(f"Running for mission phase {mission_phase}")
    mesh_file_path: Path = mesh_folder / FACET_SHAPE_MODELS[mission_phase]

    # Load original mesh arrays for libigl
    V, F = igl.read_triangle_mesh(mesh_file_path)

    # Load processed mesh tables
    points, tris = FacetParser.load_mesh(mesh_file_path)

    facets: DataFrame = data_tir_maps.filter(
        pl.col("mission_phase") == mission_phase
    )

    # ------------------------------------------------------------
    # Associate facets with mesh triangles using point-to-mesh distance
    # ------------------------------------------------------------

    P = (
        facets
        .select(["x", "y", "z"])
        .to_numpy()
        .astype(np.float64)
    )

    sqrD, tri_idx, closest_points = igl.point_mesh_squared_distance(
        P,
        V,
        F
    )

    associate_distance = np.sqrt(sqrD)

    facet_nums = (
        facets
        .get_column("facet_num")
        .to_numpy()
        .astype(np.int32)
    )

    facet_nums_to_tri_nums_lookup = pl.DataFrame(
        {
            "facet_num": facet_nums,
            "tri_num": tri_idx.astype(np.int32),
            "associate_distance": associate_distance,
        }
    )

    # Sanity checks
    if not np.all(
        facet_nums_to_tri_nums_lookup["associate_distance"].to_numpy()
        < 1e-5
    ):
        print("Facets not lining up, skipping")
        continue

    duplicates = (
        facet_nums_to_tri_nums_lookup
        .group_by("tri_num")
        .len()
        .filter(pl.col("len") > 1)
    )

    if duplicates.height != 0:
        print("duplicate triangles found!, skipping")
        print(duplicates)

        continue


    # ------------------------------------------------------------
    # Join facet measurements onto triangles
    # ------------------------------------------------------------

    tris_facets_joined: DataFrame = (
        tris
        .join(
            facet_nums_to_tri_nums_lookup,
            on="tri_num"
        )
        .join(
            facets,
            on="facet_num"
        )
    )

    if tris_facets_joined["tri_num"].n_unique() != tris_facets_joined.height:
        print("Non unique tris detected, skipping")
        continue


    # ------------------------------------------------------------
    # Create triangle-indexed measurement arrays
    # ------------------------------------------------------------

    measurement_type_triangle_values_dict: Dict[str, np.ndarray] = {}

    for measurement_type in measurement_types:

        triangle_values = np.full(
            tris.height,
            np.nan,
            dtype=np.float32,
        )

        triangle_values[
            tris_facets_joined["tri_num"].to_numpy()
        ] = (
            tris_facets_joined
            .get_column(measurement_type)
            .to_numpy()
            .astype(np.float32)
        )

        measurement_type_triangle_values_dict[measurement_type] = (
            triangle_values
        )


    # ------------------------------------------------------------
    # Rasterisation
    # ------------------------------------------------------------

    def handle_chunk(chunk: QCubeChunk) -> List[np.ndarray]:

        triangle_index_image: np.ndarray = FacetParser.rasterize_facets(
            points,
            tris,
            chunk,
        )

        measurement_types_results: List[np.ndarray] = []

        mask = ~np.isnan(triangle_index_image)

        for measurement_type in measurement_types:

            output = np.full(
                triangle_index_image.shape,
                np.nan,
                dtype=np.float32,
            )

            output[mask] = (
                measurement_type_triangle_values_dict[measurement_type]
                [
                    triangle_index_image[mask]
                    .astype(np.int32)
                ]
            )

            measurement_types_results.append(output.T)

        return measurement_types_results


    ChunkingTools.bulk_append_by_chunks(
        dp.combined_atlas.select("i", "j", "face"),
        tir_spectral_export_path,
        [f"TIR {mission_phase} {type_name}" for type_name in measurement_types],
        handle_chunk,
        chunks=QCubeChunk.generate(depth=2),
    )

c:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\.venv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Running for mission phase reconc
Facets not lining up, skipping
Running for mission phase recona


Processing chunks:   0%|          | 0/96 [00:00<?, ?it/s]

-1.4979303000000073


Processing chunks:   1%|          | 1/96 [00:03<04:57,  3.13s/it]

...

Processing chunks:   8%|▊         | 8/96 [00:14<02:23,  1.63s/it]

-0.0385707999994338


Processing chunks:   9%|▉         | 9/96 [00:16<02:36,  1.80s/it]

-0.0384066000005987


KeyboardInterrupt: 